In [89]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import cv2
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [90]:
CSV_PATH = "../annotations.csv"
IMG_DIR = "../dataset_prepro/"

In [91]:
class DatasetFaces(Dataset):   

    def __init__(self, csv_file, img_dir, transform=None, file_extension='.jpg'):
        """
        Args:
            csv_file (string): Chemin vers le fichier CSV.
            img_dir (string): Répertoire avec toutes les images.
            transform (callable, optional): Transformations à appliquer.
            file_extension (string): Extension à ajouter aux noms de fichiers
                                     (ex: '.jpg', '.png')
        """
        # --- Modification ici ---
        # On utilise 'filename' comme index du DataFrame
        self.annotations = pd.read_csv(csv_file, index_col='filename')
        
        # On stocke la liste des noms de fichiers (l'index) 
        # pour y accéder par un entier `idx`
        self.img_names = self.annotations.index.tolist()
        # --- Fin Modification ---
        
        self.img_dir = img_dir
        self.transform = transform
        self.file_extension = file_extension # Pour gérer "s1_00000" -> "s1_00000.jpg"

    def __len__(self):
        # La longueur est le nombre de noms de fichiers
        return len(self.img_names)

    def __getitem__(self, idx):
        """
        Récupère un échantillon (image + labels) à l'index donné.
        """
        
        # 1. Obtenir le nom du fichier (qui est notre index) via l'entier `idx`
        img_name_base = self.img_names[idx] # ex: "s1_00000"
        
        # 2. Construire le chemin complet de l'image
        # On ajoute l'extension de fichier que vous avez spécifiée
        img_name_with_ext = img_name_base + self.file_extension
        img_path = os.path.join(self.img_dir, img_name_with_ext)
        
        # 3. Charger l'image
        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            print(f"Erreur : Image non trouvée à {img_path}")
            # Vous pouvez retourner None ou un tenseur vide
            # pour le sauter dans le DataLoader (si vous gérez les `None`)
            return None, None 

        # 4. Obtenir TOUS les labels en utilisant le nom du fichier (l'index)
        # .loc[img_name_base] -> Récupère la ligne par son index (ex: "s1_00000")
        # .values -> Récupère les valeurs de toutes les colonnes de labels
        labels = self.annotations.loc[img_name_base].values
        
        # 5. Convertir les labels en tenseur
        # dtype=torch.float32 est souvent requis pour les fonctions de perte 
        # de classification multi-label (comme BCEWithLogitsLoss)
        labels = torch.tensor(labels.astype(float), dtype=torch.float32)
        # `labels` sera un tenseur de type [1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0]

        # 6. Appliquer les transformations
        if self.transform:
            image = self.transform(image)

        # 7. Retourner l'image et le tenseur de labels
        return image, labels
    

In [92]:
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

dataset = DatasetFaces(csv_file=CSV_PATH,
                              img_dir=IMG_DIR,
                              transform=data_transform,
                              file_extension=".png")

data_loader = DataLoader(dataset, 
                             batch_size=4, # On prend un petit lot de 4
                             shuffle=True)

In [93]:
class CNN(nn.Module):

    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,8,3,padding=1), nn.ReLU(),
            nn.Conv2d(8,16,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # Note : 32*16*16 implique que vos images d'entrée font 64x64
        # (64 -> 32 -> 16)
        self.common_fc = nn.Sequential(
            nn.Linear(32*16*16, 256), nn.ReLU(),
        )

        # --- Têtes Binaires (sortie 1 logit) ---
        # Classification de la barbe
        self.classifier_barbe = nn.Sequential(
            nn.Linear(256, 64), nn.ReLU(), # J'ajoute un ReLU ici
            nn.Linear(64, 1)
        )

        # Classification des moustaches
        self.classifier_moustache = nn.Sequential(
            nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )

        # Classification des lunettes
        self.classifier_lunettes = nn.Sequential(
            nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
        
        # --- Branche "Cheveux" ---
        
        # 1. Couche intermédiaire pour les caractéristiques des cheveux
        self.classifier_cheveux_features = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU() # J'ajoute un ReLU ici
        )

        # 2. Tête Multi-classe (sortie 3 logits)
        self.classifier_taille_cheveux = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 3) # Sortie pour 3 classes (long, court, chauve?)
        )

        # 3. Tête Multi-classe (sortie 5 logits)
        # ⚠️ CORRECTION : Renommé pour éviter l'écrasement
        self.classifier_couleur_cheveux = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 5) # Sortie pour 5 classes de couleur
        )

    def forward(self, x):
        # 1. Partie convolutive
        x = self.conv(x)
        
        # 2. Aplatir et passer dans le tronc commun
        x = x.view(x.size(0), -1) # ou torch.flatten(x, 1)
        common_features = self.common_fc(x) # Shape: [batch_size, 256]

        # 3. Têtes binaires
        out_barbe = self.classifier_barbe(common_features)
        out_moustache = self.classifier_moustache(common_features)
        out_lunettes = self.classifier_lunettes(common_features)
        
        # 4. Branche "Cheveux"
        cheveux_features = self.classifier_cheveux_features(common_features) # Shape: [batch_size, 128]
        
        # 5. Têtes multi-classes (cheveux)
        out_taille_cheveux = self.classifier_taille_cheveux(cheveux_features)
        out_couleur_cheveux = self.classifier_couleur_cheveux(cheveux_features)

        # 6. Retourner toutes les sorties (logits)
        return [out_barbe, out_moustache, out_lunettes, out_taille_cheveux, out_couleur_cheveux]

In [ ]:
from tqdm import tqdm
import torch.optim as Adam

num_epochs = 2
model = CNN() 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion_binaire = nn.BCEWithLogitsLoss().to(device)
criterion_multiclasse = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(num_epochs):
    
    print(f"--- Epoch {epoch+1}/{num_epochs} ---")
    
    # Pour suivre la perte moyenne de l'époque
    epoch_loss = 0.0

    # --- 2. "WRAPPER" LE DATA_LOADER AVEC TQDM ---
    progress_bar = tqdm(data_loader, desc="Entraînement", unit="batch")

    # --- Dans la boucle d'entraînement ---
    # 3. ITÉRER SUR LA BARRE DE PROGRESSION
    for images, all_labels_batch in progress_bar:
        
        # 1. Envoyer les tenseurs principaux sur le device
        images = images.to(device)
        all_labels_batch = all_labels_batch.to(device) # Un seul tenseur
        
        # 2. Obtenir les 5 sorties (logits) du modèle
        outputs = model(images)
        out_barbe, out_moustache, out_lunettes, out_taille, out_couleur = outputs
        
        # 3. --- PRÉPARATION DES LABELS ---
        lab_barbe = all_labels_batch[:, 0].float()
        lab_moustache = all_labels_batch[:, 1].float()
        lab_lunettes = all_labels_batch[:, 2].float()

        lab_taille_onehot = all_labels_batch[:, 3:6] 
        lab_taille = torch.argmax(lab_taille_onehot, dim=1) 
        
        lab_couleur_onehot = all_labels_batch[:, 6:11] 
        lab_couleur = torch.argmax(lab_couleur_onehot, dim=1) 

        # 4. Calculer les "loss" pour chaque tête
        loss_barbe = criterion_binaire(out_barbe.squeeze(), lab_barbe)
        loss_moustache = criterion_binaire(out_moustache.squeeze(), lab_moustache)
        loss_lunettes = criterion_binaire(out_lunettes.squeeze(), lab_lunettes)
        loss_taille = criterion_multiclasse(out_taille, lab_taille)
        loss_couleur = criterion_multiclasse(out_couleur, lab_couleur)
        
        # 5. Additionner les pertes
        total_loss = loss_barbe + loss_moustache + loss_lunettes + loss_taille + loss_couleur
        
        # 6. Rétropropagation
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        # --- 4. (BONUS) METTRE À JOUR LA BARRE AVEC LA "LOSS" ---
        epoch_loss += total_loss.item()
        progress_bar.set_postfix(loss=f"{total_loss.item():.4f}")
    
    # Afficher la perte moyenne de l'époque
    avg_loss = epoch_loss / len(data_loader)
    print(f"Loss moyenne de l'époque {epoch+1} : {avg_loss:.4f}\n")

--- Epoch 1/2 ---


Entraînement:   0%|          | 0/3973 [00:00<?, ?batch/s, loss=4.7956]

Entraînement:  85%|████████▍ | 3377/3973 [01:24<00:15, 38.00batch/s, loss=1.1277]

In [ ]:
# 1. Définir les classes (selon l'ordre de vos colonnes CSV)
# Ordre présumé : Long, Court, Chauve
CLASSES_TAILLE = {0: 'Longs', 1: 'Courts', 2: 'Chauve'}

# Ordre présumé : Blonds, Chatains, Roux, Brun, Gris_bleu
CLASSES_COULEUR = {0: 'Blond', 1: 'Chatain', 2: 'Roux', 3: 'Brun', 4: 'Gris/Bleu'}

def predict_single_image(image_path, model, device):
    """
    Charge une image, effectue la prédiction et affiche les résultats.
    """
    
    # --- A. Préparation de l'image ---
    
    # Charger l'image avec PIL
    try:
        img_raw = Image.open(image_path).convert('RGB')
    except FileNotFoundError:
        print(f"❌ Erreur : Image introuvable à {image_path}")
        return

    # Appliquer les transformations
    img_tensor = data_transform(img_raw)
    
    # ⚠️ CRUCIAL : Ajouter la dimension du batch (3, 224, 224) -> (1, 3, 224, 224)
    img_tensor = img_tensor.unsqueeze(0)
    
    # Envoyer sur le device (GPU/CPU)
    img_tensor = img_tensor.to(device)
    
    # --- B. Inférence ---
    model.eval() # Mettre le modèle en mode évaluation (fige le Dropout/BatchNorm)
    
    with torch.no_grad(): # Désactiver le calcul des gradients (économie mémoire)
        outputs = model(img_tensor)
        
    # Récupérer les 5 sorties
    logit_barbe, logit_moustache, logit_lunettes, logit_taille, logit_couleur = outputs
    
    # --- C. Décodage des résultats ---
    
    # 1. Têtes Binaires : On applique Sigmoid
    # Si proba > 0.5 -> Oui (True), sinon Non (False)
    prob_barbe = torch.sigmoid(logit_barbe).item()
    prob_moustache = torch.sigmoid(logit_moustache).item()
    prob_lunettes = torch.sigmoid(logit_lunettes).item()
    
    res_barbe = "Oui" if prob_barbe > 0.5 else "Non"
    res_moustache = "Oui" if prob_moustache > 0.5 else "Non"
    res_lunettes = "Oui" if prob_lunettes > 0.5 else "Non"

    # 2. Têtes Multi-classes : On applique Softmax puis Argmax
    
    # Taille
    prob_taille = torch.softmax(logit_taille, dim=1)
    idx_taille = torch.argmax(prob_taille, dim=1).item()
    res_taille = CLASSES_TAILLE[idx_taille]
    confiance_taille = prob_taille[0][idx_taille].item() # Probabilité de la classe gagnante
    
    # Couleur
    prob_couleur = torch.softmax(logit_couleur, dim=1)
    idx_couleur = torch.argmax(prob_couleur, dim=1).item()
    res_couleur = CLASSES_COULEUR[idx_couleur]
    confiance_couleur = prob_couleur[0][idx_couleur].item()

    # --- D. Affichage ---
    print(f"\n🔍 ANALYSE DE : {image_path}")
    print("-" * 30)
    print(f"🧔 Barbe      : {res_barbe} \t({prob_barbe:.1%})")
    print(f"👨 Moustache  : {res_moustache} \t({prob_moustache:.1%})")
    print(f"👓 Lunettes   : {res_lunettes} \t({prob_lunettes:.1%})")
    print(f"💇 Taille     : {res_taille} \t(confiance : {confiance_taille:.1%})")
    print(f"🎨 Couleur    : {res_couleur} \t(confiance : {confiance_couleur:.1%})")
    print("-" * 30)
    
    # Afficher l'image
    plt.imshow(img_raw)
    plt.axis('off')
    plt.title(f"{res_taille}, {res_couleur}")
    plt.show()

# --- Utilisation ---
# Assurez-vous que 'model' est votre modèle entraîné et 'device' est défini
chemin_image_test = "../dataset_prepro/s1_00001.png" # Mettez un vrai chemin ici

predict_single_image(chemin_image_test, model, device)